O primeiro desafio é de diagnóstico livre. Vocês vão usar os comandos de exploração que aprendemos na terça — info, describe, nunique, isnull — sem que eu diga o que procurar. Essa é a situação real: o RH mandou a base, ninguém vai te dizer "roda o info". Você decide o que investigar.
Tem três perguntas para responder no Markdown logo abaixo do código: quantas colunas são numéricas e quantas são texto, se existe nulo em alguma coluna, e qual coluna tem o maior número de valores únicos e o que isso diz sobre ela. Acostumem com essa prática de escrever a análise em texto — dado sem explicação não serve para ninguém.

O Desafio 2 é o mais parecido com o dia a dia de trabalho. O gestor de RH fez três perguntas e vocês precisam responder com código. Vão precisar de filtro composto, groupby com média e groupby agrupando por dois campos ao mesmo tempo.
O ponto onde mais gente erra é o filtro composto. Cada condição precisa estar entre parênteses, e usa o & para "e". Sem os parênteses o Python não interpreta certo e ou dá erro ou retorna resultado errado.

O Desafio 3 envolve datas — conteúdo da terça. A sequência é sempre a mesma: primeiro converter com pd.to_datetime, depois usar o acessor .dt para extrair o que precisar. Se você tentar usar o .dt.year antes de converter, o pandas dá erro porque a coluna ainda é texto.
Uma das perguntas é sobre funcionários ativos com 60 anos ou mais — atenção especial para esse RH, são pessoas próximas da aposentadoria. A outra é sobre o histórico de admissões por ano: groupby no Ano_Admissao e .size() para contar. O sort_index() no final ordena por ano em vez de por quantidade.

DESAFIO 1

In [20]:
import pandas as pd
import matplotlib.pyplot as plt

URL = r"C:\Users\leogo\OneDrive\Documentos\SCTEC\GitHub\turma-visualizacao-de-dados\alunos\Leo_Gobel\Semana_04\base_rh_por_depto.xlsx"

df = pd.read_excel(URL)

In [22]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID_Funcionario  1000 non-null   int64  
 1   Nome            1000 non-null   str    
 2   Departamento    1000 non-null   str    
 3   Cargo           1000 non-null   str    
 4   Salario         1000 non-null   float64
 5   Data_Admissao   1000 non-null   str    
 6   Genero          1000 non-null   str    
 7   Idade           1000 non-null   int64  
 8   Estado_Civil    1000 non-null   str    
 9   Status          1000 non-null   str    
dtypes: float64(1), int64(2), str(7)
memory usage: 78.3 KB


In [23]:
df.describe().round(2)

,ID_Funcionario,Salario,Idade
count,1000.00,1000.00,1000.00
mean,500.50,8579.95,41.40
std,288.82,3657.37,13.67
min,1.00,2000.71,18.00
25%,250.75,5564.54,30.00
50%,500.50,8571.13,41.00
75%,750.25,11554.63,53.00
max,1000.00,14954.51,65.00


In [24]:
df.isnull().sum()

ID_Funcionario    0
Nome              0
Departamento      0
Cargo             0
Salario           0
Data_Admissao     0
Genero            0
Idade             0
Estado_Civil      0
Status            0
dtype: int64

In [25]:
df.nunique().sort_values(ascending=False)

ID_Funcionario    1000
Salario           1000
Nome               976
Data_Admissao      878
Idade               48
Departamento         6
Cargo                5
Estado_Civil         4
Genero               2
Status               2
dtype: int64

DESAFIO 2

In [ ]:
df["Data_Admissao"] = pd.to_datetime(
    df["Data_Admissao"],
    format="%d/%m/%Y",
    errors="coerce"
)

print(f"Dataset carregado: {df.shape[0]} linhas x {df.shape[1]} colunas")

df['Departamento'] = df['Departamento'].astype('category')
df['Status'] = df['Status'].astype('category')



Dataset carregado: 1000 linhas x 10 colunas


In [5]:
filtro_ativos_ti = df[(df["Status"] == "Ativo") & (df["Departamento"] == "TI")]
print(filtro_ativos_ti)


     ID_Funcionario                      Nome Departamento        Cargo  \
18               19       Bernardo Cavalcanti           TI  Coordenador   
36               37        Dra. Letícia Jesus           TI   Assistente   
39               40        Srta. Amanda Souza           TI     Analista   
55               56        Maria Julia Araújo           TI   Assistente   
59               60           Dr. André Ramos           TI      Técnico   
..              ...                       ...          ...          ...   
914             915              Davi da Mota           TI     Analista   
927             928         Valentina Cardoso           TI      Gerente   
930             931             Yuri Ferreira           TI  Coordenador   
951             952              Calebe Alves           TI  Coordenador   
955             956  Sra. Gabriela Cavalcanti           TI   Assistente   

      Salario Data_Admissao Genero  Idade Estado_Civil Status  
18    6088.45    2020-02-06      F 

In [13]:
media_salarial_depto = df.groupby("Departamento")["Salario"].mean().round(2)
print(media_salarial_depto)


Departamento
Financeiro    8333.12
Logística     8881.66
Produção      8968.73
RH            8791.58
TI            8142.91
Vendas        8317.08
Name: Salario, dtype: float64


In [14]:
resumo_cruzado = df.groupby(["Departamento", "Cargo"])["Salario"].mean().round(2).reset_index()
print(resumo_cruzado)

   Departamento        Cargo  Salario
0    Financeiro     Analista  8022.34
1    Financeiro   Assistente  8635.77
2    Financeiro  Coordenador  8385.92
3    Financeiro      Gerente  8129.22
4    Financeiro      Técnico  8606.00
5     Logística     Analista  8248.20
6     Logística   Assistente  9210.18
7     Logística  Coordenador  9251.97
8     Logística      Gerente  8553.65
9     Logística      Técnico  9028.27
10     Produção     Analista  9620.57
11     Produção   Assistente  9936.56
12     Produção  Coordenador  7533.27
13     Produção      Gerente  9307.70
14     Produção      Técnico  8658.23
15           RH     Analista  8440.43
16           RH   Assistente  8288.23
17           RH  Coordenador  9826.40
18           RH      Gerente  9167.42
19           RH      Técnico  7673.44
20           TI     Analista  8555.98
21           TI   Assistente  8655.15
22           TI  Coordenador  8438.68
23           TI      Gerente  6958.66
24           TI      Técnico  8231.03
25       Ven

DESAFIO 3 

In [16]:
proximos_aposentadoria = df[(df["Status"] == "Ativo") & (df["Idade"] >= 60)]

print(f"Total de funcionários ativos com 60+ anos: {len(proximos_aposentadoria)}")
print(proximos_aposentadoria[["Nome", "Idade", "Departamento", "Cargo"]])

Total de funcionários ativos com 60+ anos: 54
                          Nome  Idade Departamento        Cargo
12                Laís da Mata     64       Vendas     Analista
14               Laura Rezende     65    Logística     Analista
45               Murilo da Paz     62    Logística     Analista
57             Nina Cavalcanti     60   Financeiro      Técnico
66       João Gabriel Caldeira     63     Produção   Assistente
78                João Barbosa     65           TI   Assistente
80            Gabriela Freitas     65    Logística     Analista
108  Dra. Maria Alice Oliveira     63           TI      Técnico
112          Pedro Lucas Cunha     65   Financeiro   Assistente
118             Pietro Almeida     60       Vendas  Coordenador
119         Davi Lucca Cardoso     64    Logística      Gerente
157            Isadora Azevedo     64       Vendas      Técnico
191             Laís Rodrigues     64   Financeiro      Técnico
233                Clara Cunha     64    Logística      Té

In [19]:
admissoes_ano = (
    df.groupby('Ano_Admissao')
      .size()
      .sort_index()     # cronológico — não por quantidade
)

print(admissoes_ano)

# Bônus: qual ano teve mais admissões?
print(f"\nPico: {admissoes_ano.idxmax()} ({admissoes_ano.max()} admissões)")

Ano_Admissao
2015     33
2016     87
2017     99
2018     89
2019    101
2020     94
2021     94
2022    108
2023    106
2024    113
2025     76
dtype: int64

Pico: 2024 (113 admissões)
